# 07 · Doctor Strange's time stone
*"Which version of this record was true last Tuesday?"*

Delta keeps a versioned log of every change. You will tamper with a table (Hydra!), see exactly what changed,
and restore it. Needs chapter 06's `shield.silver.incident_reports`.

In [ ]:
T = "shield.silver.incident_reports"

## Turn on the change data feed

In [ ]:
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
good_version = spark.sql(f"DESCRIBE HISTORY {T}").first()["version"]
print("Known-good version:", good_version)

## Hydra tampers: downgrade critical incidents

In [ ]:
spark.sql(f"UPDATE {T} SET severity = 'low' WHERE severity = 'critical'")
display(spark.sql(f"SELECT severity, COUNT(*) AS n FROM {T} GROUP BY severity"))

## Audit: what changed, when, and by whom?

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}").select("version", "timestamp", "userName", "operation", "operationMetrics"))

### Row-level view with the change data feed

In [ ]:
changes = spark.sql(f"SELECT * FROM table_changes('{T}', {good_version + 1})")
display(changes.groupBy("_change_type").count())
display(changes.filter("_change_type = 'update_preimage'").select("incident_id", "severity").limit(5))

## Time travel: read the past

In [ ]:
before = spark.sql(f"SELECT severity, COUNT(*) AS n FROM {T} VERSION AS OF {good_version} GROUP BY severity")
display(before)

## Undo it

In [ ]:
spark.sql(f"RESTORE TABLE {T} TO VERSION AS OF {good_version}")
display(spark.sql(f"SELECT severity, COUNT(*) AS n FROM {T} GROUP BY severity"))

`RESTORE` is itself a new version, so the tampering stays visible in the history. Nothing is erased.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {T}").select("version", "operation").limit(5))

## Try it yourself
1. Use `TIMESTAMP AS OF` instead of a version number.
2. Use the change feed to build `silver.incident_audit` (incident_id, old severity, new severity, changed_at).
3. What happens to time travel after `VACUUM ... RETAIN 0 HOURS`? Try it on a throwaway table only.